In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

import optuna

In [12]:
df = pd.read_csv("datasets/kingametric_credit_risk.csv")

In [13]:
df.shape

(8744, 45)

In [14]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

for col in df.select_dtypes('number').columns: 
    Q1=df[col].quantile(0.25); 
    Q3=df[col].quantile(0.75); 
    IQR=Q3-Q1; 
    
    df[col]=df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)

df.fillna(df.median(numeric_only=True), inplace=True)

In [15]:
cat_cols = ["Payment_of_Min_Amount", "Credit_Mix", "Payment_Behaviour", "Borrower_Tier"]

df[cat_cols] = df[cat_cols].astype("category")

In [16]:
X = df.drop(columns=["Default_Flag"], axis=1)
y = df["Default_Flag"]

In [17]:
X.shape

(8744, 44)

In [18]:
corr = X.corr(numeric_only=True).abs()

upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]

X = X.drop(columns=to_drop)

In [19]:
X.shape

(8744, 33)

In [20]:
folds = StratifiedKFold(n_splits=5, shuffle=True,random_state=42)

In [21]:
base_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    enable_categorical=True, 
    tree_method="hist",
    learning_rate=0.05,
    random_state=42
)
base_model.fit(X, y)

importances = base_model.feature_importances_

feature_importance = (
    pd.Series(importances, index=X.columns).sort_values(ascending=False)
)

In [22]:
top_features = feature_importance.head(30).index

X_select = X[top_features]

In [23]:
def objective(trial):
    
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "subsample": trial.suggest_float("subsample", 0.7, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 0.95),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 0.5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 3),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 5),
        "eval_metric": "auc",
        "tree_method": "hist",
        "enable_categorical":True,
        "random_state": 42
    }

    auc_scores = []

    for train_idx, val_idx in folds.split(X_select, y):
        
        X_train, X_val = X_select.iloc[train_idx], X_select.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBClassifier(**params,)

        model.fit(X_train, y_train)

        y_pred = model.predict_proba(X_val)[:,1]

        auc = roc_auc_score(y_val, y_pred)
        auc_scores.append(auc)

    return np.mean(auc_scores)

In [ ]:
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=5,  
    interval_steps=1     
)

study = optuna.create_study(direction="maximize", pruner=pruner)

study.optimize(objective, n_trials=100)

[I 2026-03-23 06:21:22,230] A new study created in memory with name: no-name-dbc9775a-ae58-4e71-b3e1-957fc2ab6950
[I 2026-03-23 06:21:27,961] Trial 0 finished with value: 0.6267151893974972 and parameters: {'n_estimators': 771, 'max_depth': 6, 'learning_rate': 0.01668081470608862, 'subsample': 0.7978099915371262, 'colsample_bytree': 0.8049742469300735, 'min_child_weight': 2, 'gamma': 0.1346118801221608, 'reg_alpha': 0.8905574259177245, 'reg_lambda': 1.0308076889022235, 'scale_pos_weight': 3.016719744357352}. Best is trial 0 with value: 0.6267151893974972.
[I 2026-03-23 06:21:31,594] Trial 1 finished with value: 0.6079310510092304 and parameters: {'n_estimators': 594, 'max_depth': 6, 'learning_rate': 0.059864411557244784, 'subsample': 0.8079907919829087, 'colsample_bytree': 0.847751276247822, 'min_child_weight': 4, 'gamma': 0.14048945967921295, 'reg_alpha': 0.3429267629729009, 'reg_lambda': 0.83469255530656, 'scale_pos_weight': 4.367834599070081}. Best is trial 0 with value: 0.626715189

In [ ]:
best_params = study.best_params

print(best_params)

{'n_estimators': 370, 'max_depth': 4, 'learning_rate': 0.01049846784508983, 'subsample': 0.804578691117884, 'colsample_bytree': 0.7589769081528248, 'min_child_weight': 1, 'gamma': 0.16850897198346368, 'reg_alpha': 0.9001990568606176, 'reg_lambda': 1.640042344449604, 'scale_pos_weight': 3.5347720768787916}


In [ ]:
fin_model = XGBClassifier(**best_params, enable_categorical=True, tree_method="hist")

fin_model.fit(X_select, y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.7589769081528248
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [ ]:
auc_scores = []

for train_idx, val_idx in folds.split(X_select, y):

    fin_model.fit(X_select.iloc[train_idx], y.iloc[train_idx])
    preds = fin_model.predict_proba(X_select.iloc[val_idx])[:,1]

    auc_scores.append(roc_auc_score(y.iloc[val_idx], preds))

print("Final AUC", np.mean(auc_scores))

Final AUC 0.6453906067391095
